In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# CHSH 부등식

*사용 시간 추정: Heron r2 프로세서에서 약 2분 (참고: 이는 추정치이며, 실제 실행 시간은 다를 수 있습니다.)*
## 배경
이 튜토리얼에서는 Estimator 프리미티브를 사용하여 양자 컴퓨터에서 CHSH 부등식의 위반을 증명하는 실험을 실행합니다.

CHSH 부등식은 저자인 Clauser, Horne, Shimony, Holt의 이름을 따서 명명되었으며, 벨의 정리(1969)를 실험적으로 증명하는 데 사용됩니다. 이 정리는 국소 숨은 변수 이론이 양자역학에서 얽힘의 일부 결과를 설명할 수 없다고 주장합니다. CHSH 부등식의 위반은 양자역학이 국소 숨은 변수 이론과 양립할 수 없음을 보여주는 데 사용됩니다. 이는 양자역학의 기초를 이해하는 데 중요한 실험입니다.

2022년 노벨 물리학상은 Alain Aspect, John Clauser, Anton Zeilinger에게 수여되었으며, 이는 부분적으로 양자 정보 과학에서의 선구적인 연구, 특히 얽힌 광자를 이용하여 벨 부등식의 위반을 증명한 실험에 대한 공로를 인정한 것입니다.
## 요구 사항
이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요:

* Qiskit SDK v1.0 이상 ([시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 이상
## 설정

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

## 1단계: 고전적 입력을 양자 문제로 매핑하기
이 실험에서는 두 Qubit를 얽은 쌍을 생성하고, 각 Qubit를 두 가지 다른 기저에서 측정합니다. 첫 번째 Qubit의 기저를 $A$와 $a$, 두 번째 Qubit의 기저를 $B$와 $b$로 레이블합니다. 이를 통해 CHSH 양 $S_1$을 계산할 수 있습니다:

$$
S_1 = A(B-b) + a(B+b).
$$

각 관측량은 $+1$ 또는 $-1$입니다. 분명히, $B\pm b$ 항 중 하나는 $0$이어야 하고, 다른 하나는 $\pm 2$이어야 합니다. 따라서 $S_1 = \pm 2$입니다. $S_1$의 평균값은 다음 부등식을 만족해야 합니다:

$$
|\langle S_1 \rangle|\leq 2.
$$

$S_1$을 $A$, $a$, $B$, $b$로 전개하면 다음과 같습니다:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2
$$

또 다른 CHSH 양 $S_2$를 정의할 수 있습니다:

$$
S_2 = A(B+b) - a(B-b),
$$

이는 또 다른 부등식으로 이어집니다:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2
$$

양자역학이 국소 숨은 변수 이론으로 설명될 수 있다면, 위의 부등식이 성립해야 합니다. 그러나 이 튜토리얼에서 보여주듯이, 이 부등식은 양자 컴퓨터에서 위반될 수 있습니다. 따라서 양자역학은 국소 숨은 변수 이론과 양립할 수 없습니다.
더 많은 이론을 배우고 싶다면, John Watrous와 함께하는 [행동 속의 얽힘](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game)을 살펴보세요.
양자 컴퓨터에서 두 Qubit 사이에 얽힌 쌍을 생성하기 위해 벨 상태 $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$를 만듭니다. Estimator 프리미티브를 사용하면, 두 CHSH 양 $\langle S_1\rangle$과 $\langle S_2\rangle$의 기댓값을 계산하는 데 필요한 기댓값($\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$, $\langle ab \rangle$)을 직접 구할 수 있습니다. Estimator 프리미티브가 도입되기 전에는 측정 결과로부터 기댓값을 직접 구성해야 했습니다.

두 번째 Qubit를 $Z$ 기저와 $X$ 기저에서 측정합니다. 첫 번째 Qubit도 직교 기저에서 측정하지만, 두 번째 Qubit에 대한 각도를 $0$에서 $2\pi$ 사이로 스윕합니다. 보시다시피, Estimator 프리미티브를 사용하면 매개변수화된 Circuit 실행이 매우 간편합니다. 일련의 CHSH Circuit을 생성하는 대신, 측정 각도를 지정하는 매개변수와 해당 매개변수의 위상 값 목록을 가진 *하나의* CHSH Circuit만 생성하면 됩니다.

마지막으로, 결과를 분석하고 측정 각도에 대해 플롯합니다. 특정 측정 각도 범위에서 CHSH 양의 기댓값 $|\langle S_1\rangle| > 2$ 또는 $|\langle S_2\rangle| > 2$가 됨을 확인할 수 있으며, 이는 CHSH 부등식의 위반을 증명합니다.

In [2]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_kingston'

### Create a parameterized CHSH circuit

First, we write the circuit with the parameter $\theta$, which we call `theta`. The [`Estimator` primitive](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) can enormously simplify circuit building and output analysis by directly providing expectation values of observables. Many problems of interest, especially for near-term applications on noisy systems, can be formulated in terms of expectation values. `Estimator` (V2) primitive can automatically change measurement basis based on the supplied observable.

In [3]:
theta = Parameter("$\\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif" alt="Output of the previous code cell" />

### 매개변수화된 CHSH Circuit 생성하기
먼저, `theta`라고 부르는 매개변수 $\theta$를 포함한 Circuit을 작성합니다. [`Estimator` 프리미티브](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2)는 관측량의 기댓값을 직접 제공하여 Circuit 구성 및 출력 분석을 크게 단순화할 수 있습니다. 관심 있는 많은 문제, 특히 노이즈가 있는 시스템의 단기 응용 프로그램에서는 기댓값으로 공식화할 수 있습니다. `Estimator` (V2) 프리미티브는 제공된 관측량에 따라 측정 기저를 자동으로 변경할 수 있습니다.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as list of lists in order to work
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif)

### 나중에 할당할 위상 값 목록 생성하기
매개변수화된 CHSH Circuit을 생성한 후, 다음 단계에서 Circuit에 할당할 위상 값 목록을 생성합니다. 다음 코드를 사용하여 $0$에서 $2 \pi$까지 균등한 간격으로 21개의 위상 값 목록, 즉 $0$, $0.1 \pi$, $0.2 \pi$, ..., $1.9 \pi$, $2 \pi$를 생성할 수 있습니다.

In [5]:
# <CHSH1> = <AB> - <Ab> + <aB> + <ab> -> <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <CHSH2> = <AB> + <Ab> - <aB> + <ab> -> <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

### 관측량
이제 기댓값을 계산하기 위한 관측량이 필요합니다. 이 경우 각 Qubit의 직교 기저를 살펴보면서, 첫 번째 Qubit의 매개변수화된 $Y$ 회전이 두 번째 Qubit 기저에 대해 측정 기저를 거의 연속적으로 스윕합니다. 따라서 관측량 $ZZ$, $ZX$, $XZ$, $XX$를 선택합니다.

In [6]:
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)

chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif" alt="Output of the previous code cell" />

## 2단계: 양자 하드웨어 실행을 위한 문제 최적화

총 작업 실행 시간을 줄이기 위해, V2 프리미티브는 대상 시스템이 지원하는 명령어 및 연결성에 부합하는 Circuit과 관측량만 허용합니다 (이를 명령어 집합 아키텍처(ISA) Circuit 및 관측량이라고 합니다).

### ISA Circuit

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif)

### ISA 관측량

마찬가지로, [`Runtime Estimator V2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run)로 작업을 실행하기 전에 관측량을 Backend와 호환되도록 변환해야 합니다. `SparsePauliOp` 객체의 `apply_layout` 메서드를 사용하여 변환을 수행할 수 있습니다.

In [8]:
# To run on a local simulator:
# Use the StatevectorEstimator from qiskit.primitives instead.

estimator = Estimator(mode=backend)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA Observables
    individual_phases,  # Parameter values
)

job_result = estimator.run(pubs=[pub]).result()

## 3단계: Qiskit 프리미티브를 사용하여 실행하기
[`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2)를 한 번 호출하여 전체 실험을 실행하기 위해,
[Qiskit Runtime `Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) 프리미티브를 생성하여 기댓값을 계산할 수 있습니다. `EstimatorV2.run()` 메서드는 `primitive unified blocs (PUBs)`의 이터러블을 받습니다. 각 PUB는 `(circuit, observables, parameter_values: Optional, precision: Optional)` 형식의 이터러블입니다.

In [9]:
chsh1_est = job_result[0].data.evs[0]
chsh2_est = job_result[0].data.evs[1]

In [10]:
fig, ax = plt.subplots(figsize=(10, 6))

# results from hardware
ax.plot(phases / np.pi, chsh1_est, "o-", label="CHSH1", zorder=3)
ax.plot(phases / np.pi, chsh2_est, "o-", label="CHSH2", zorder=3)

# classical bound +-2
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")

# quantum bound, +-2√2
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

# set x tick labels to the unit of pi
ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

# set labels, and legend
plt.xlabel("Theta")
plt.ylabel("CHSH witness")
plt.legend()
plt.show()

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/f6267448-0.avif" alt="Output of the previous code cell" />

In the figure, the lines and gray areas delimit the bounds; the outer-most (dash-dotted) lines delimit the quantum-bounds ($\pm 2$), whereas the inner (dashed) lines delimit the classical bounds ($\pm 2\sqrt{2}$). You can see that there are regions where the CHSH witness quantities exceeds the classical bounds. Congratulations! You have successfully demonstrated the violation of CHSH inequality in a real quantum system!

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3xxAgm1SF1wGp9k)